# SUMO RL — Multi-Agent Traffic Light Control

**QR-DQN multi-ágens közlekedési lámpa vezérlés SUMO szimulátorral**

Ez a notebook mindent beállít és elindít a Google Colab-ban.

---

## Gyors összefoglaló
- **21 irányított csomópont** (junction), mindegyiknek saját QR-DQN ágens
- **Reward**: log-sigmoid normalizált waiting time + CO2, [0, 1] tartományban
- **Forgalom**: lane-szintű random generálás, sumolib-bal
- **Epizód hossz**: rotáció [900, 1800, 2700, 3600, 5400, 7200] másodperc
- **Logging**: Weights & Biases

## 1. Repo klónozás + Setup

In [ ]:
# Repo klónozás
!git clone https://github.com/wagnertamas/intergreen_matrix_calculator.git
%cd intergreen_matrix_calculator

In [ ]:
# SUMO + összes Python dependency telepítése (~2-3 perc)
!chmod +x colab_setup.sh && ./colab_setup.sh

In [ ]:
# Környezeti változók beállítása (minden cellában érvényes legyen)
import os
os.environ['SUMO_HOME'] = '/usr/share/sumo'
os.environ['USE_LIBSUMO'] = '1'

## 2. WandB bejelentkezés

API key-t a https://wandb.ai/authorize oldalon találod.

In [ ]:
import wandb
wandb.login()

## 3a. Egyszerű tanítás (Single Run)

In [ ]:
# --- Paraméterek ---
PROJECT = "sumo-rl-single"  # WandB projekt név
TIMESTEPS = 100000           # Tanítási lépések (50k = gyors teszt, 500k+ = valós)

!USE_LIBSUMO=1 python main_headless.py \
    --config training_config.yaml \
    --timesteps {TIMESTEPS} \
    --project {PROJECT}

## 3b. WandB Sweep (Hyperparameter Search)

Bayesian optimalizáció a legjobb hiperparaméterekért.

In [ ]:
# --- Sweep létrehozás ---
# Ez kiírja a sweep ID-t, pl: "Created sweep with ID: abc123"
# Másold be az alábbi SWEEP_ID változóba!
!wandb sweep data/sweep_config.yaml --project sumo-rl-sweep

In [ ]:
# --- Sweep futtatás ---
# Cseréld ki a SWEEP_ID-t a fenti output alapján!
SWEEP_ID = "YOUR_ENTITY/sumo-rl-sweep/SWEEP_ID_HERE"  # <-- IDE MÁSOLD

os.environ['SWEEP_PROJECT'] = 'sumo-rl-sweep'
os.environ['SWEEP_TIMESTEPS'] = '100000'

!USE_LIBSUMO=1 wandb agent {SWEEP_ID} --count 10  # 10 trial

## 3c. Transfer Learning

Korábban tanított modell finomhangolása.

In [ ]:
# --- Ha WandB-ről töltöd le a modellt ---
# MODEL_PATH = "runs/best_model.zip"  # helyi útvonal
# !USE_LIBSUMO=1 python main_headless.py \
#     --config training_config.yaml \
#     --load-model {MODEL_PATH} \
#     --timesteps 50000 \
#     --project sumo-rl-finetune

## 4. Eredmények ellenőrzése

In [ ]:
# Mentett modellek listázása
import glob
models = glob.glob('runs/**/*.zip', recursive=True)
print(f"Mentett modellek ({len(models)}):")
for m in sorted(models):
    size_mb = os.path.getsize(m) / (1024*1024)
    print(f"  {m} ({size_mb:.1f} MB)")

In [ ]:
# TensorBoard (opcionális — WandB-n is megnézheted)
%load_ext tensorboard
%tensorboard --logdir runs/

## 5. Modell letöltése Colab-ból

A tanítás után a `.zip` modelleket letöltheted.

In [ ]:
# Google Drive-ra mentés (ajánlott hosszú tanításnál)
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r runs/ /content/drive/MyDrive/sumo_rl_runs/

In [ ]:
# Közvetlen letöltés a böngészőbe
# from google.colab import files
# files.download('runs/best_model.zip')

---

## Tippek

| Szituáció | Parancs |
|---|---|
| Gyors teszt | `TIMESTEPS = 10000` |
| Éles tanítás | `TIMESTEPS = 500000` |
| GPU ellenőrzés | `!nvidia-smi` |
| Runtime típus | Runtime → Change runtime type → GPU |
| Háttérben futás | Colab Pro-ban: Runtime → Managed sessions |
| WandB dashboard | https://wandb.ai/ |